# Multivariable Optimization

Companion notebook for the [Multivariable Optimization](https://ml-viz-ruby.vercel.app/courses/calculus-for-ml/03-multivariable-optimization) lesson.

We work the same example as the lesson, $f(x, y) = x^2 + xy + y^2 - 3x$, end to end:

1. Find the critical point by solving $\nabla f = 0$.
2. Build the Hessian and verify it with finite differences.
3. Classify the critical point from the Hessian's eigenvalues.
4. Take one **Newton step** and watch it land exactly on the minimum, then compare to a gradient step.
5. Run the eigenvalue convexity test, and study how the learning rate relates to $\lambda_{\max}$.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

plt.style.use('dark_background')
plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#30344a', 'text.color': '#e2e8f0',
    'axes.labelcolor': '#e2e8f0', 'xtick.color': '#94a3b8', 'ytick.color': '#94a3b8',
})

## The worked example: $f(x,y) = x^2 + xy + y^2 - 3x$

We define the function, its analytic gradient, and its analytic Hessian.

$$\nabla f = \begin{bmatrix} 2x + y - 3 \\ x + 2y \end{bmatrix},
\qquad
\mathbf{H} = \begin{bmatrix} 2 & 1 \\ 1 & 2 \end{bmatrix}.$$

The Hessian is constant because $f$ is quadratic.

In [ ]:
def f(p):
    x, y = p
    return x**2 + x*y + y**2 - 3*x

def grad_f(p):
    x, y = p
    return np.array([2*x + y - 3, x + 2*y])

def hessian_f(p):
    # Constant Hessian for this quadratic; argument kept for a general API.
    return np.array([[2., 1.],
                     [1., 2.]])

# --- Step 1: solve grad_f = 0 for the critical point ---
# 2x + y - 3 = 0 and x + 2y = 0  ->  H @ [x, y] = [3, 0]
A = np.array([[2., 1.],
              [1., 2.]])
b = np.array([3., 0.])
critical = np.linalg.solve(A, b)
print("Critical point (solved grad_f = 0): {}".format(critical))
print("grad_f at critical point:          {}".format(grad_f(critical)))
print("f at critical point:               {:.4f}".format(f(critical)))


### Verify the Hessian with finite differences

The analytic Hessian above should match a numerical estimate. We approximate each
second partial with the central-difference stencil

$$H_{ij} \approx \frac{f(\mathbf{x}+h e_i + h e_j) - f(\mathbf{x}+h e_i - h e_j) - f(\mathbf{x}-h e_i + h e_j) + f(\mathbf{x}-h e_i - h e_j)}{4h^2}.$$

In [ ]:
def classify_critical_point(H):
    eigs = np.linalg.eigvalsh(H)  # eigvalsh: H is symmetric, eigenvalues are real
    if np.all(eigs > 0):  return 'Local minimum (all lambda > 0)'
    if np.all(eigs < 0):  return 'Local maximum (all lambda < 0)'
    if np.all(eigs == 0): return 'Degenerate'
    return 'Saddle point (mixed lambda signs)'

# Our worked example first, then the canonical min / saddle / max trio.
hessians = {
    'f=x^2+xy+y^2-3x  at (2,-1)': hessian_f(critical),  # the lesson example
    'f=x^2+y^2        at (0,0)':  np.array([[2., 0.], [0., 2.]]),
    'f=x^2-y^2        at (0,0)':  np.array([[2., 0.], [0., -2.]]),
    'f=-x^2-y^2       at (0,0)':  np.array([[-2., 0.], [0., -2.]]),
}

for name, H in hessians.items():
    eigs = np.linalg.eigvalsh(H)
    print(name)
    print('  Eigenvalues:    {}'.format(np.round(eigs, 4)))
    print('  Classification: {}\n'.format(classify_critical_point(H)))


## One Newton step vs. one gradient step

Newton's method minimizes the second-order Taylor model exactly:

$$\mathbf{s} = -\mathbf{H}^{-1}\nabla f(\mathbf{x}_0), \qquad \mathbf{x} \leftarrow \mathbf{x}_0 + \mathbf{s}.$$

Starting from $\mathbf{x}_0 = (0,0)$ we have $\nabla f = (-3, 0)$, and the single Newton
step should land exactly on the minimum $(2, -1)$ because $f$ is quadratic. A plain
gradient step only crawls along the gradient direction.

In [ ]:
x0 = np.array([0.0, 0.0])
g0 = grad_f(x0)
H0 = hessian_f(x0)

# Newton step: solve H s = -g  (equivalent to s = -inv(H) @ g, but solve is stabler)
newton_step = np.linalg.solve(H0, -g0)
x_newton = x0 + newton_step

# One gradient step at a sensible learning rate (eta = 1 / lambda_max).
eta = 1.0 / np.max(np.linalg.eigvalsh(H0))
x_grad = x0 - eta * g0

print("Start x0:          {}".format(x0))
print("grad at x0:        {}".format(g0))
print("Newton step s:     {}".format(newton_step))
print("After Newton step: {}   f = {:.4f}".format(x_newton, f(x_newton)))
print("After grad step:   {}   f = {:.4f}   (eta = {:.3f})".format(x_grad, f(x_grad), eta))
print("True minimum:      {}   f = {:.4f}".format(critical, f(critical)))
print("\nNewton landed on the minimum exactly? {}".format(
    np.allclose(x_newton, critical)))

# Now iterate gradient descent to show how many steps it needs to catch up.
w = x0.copy()
for k in range(1, 1001):
    w = w - eta * grad_f(w)
    if np.linalg.norm(w - critical) < 1e-6:
        print("Gradient descent reached the minimum after {} steps.".format(k))
        break


## The eigenvalue convexity test

A quadratic $f(\mathbf{x}) = \tfrac12 \mathbf{x}^\top \mathbf{H}\mathbf{x}$ is convex iff
$\mathbf{H} \succeq 0$, i.e. every eigenvalue is $\geq 0$. We reproduce the two
matrices worked by hand in the lesson.

In [ ]:
def is_convex_quadratic(H):
    """A quadratic x^T H x is convex iff H is positive semi-definite."""
    eigenvalues = np.linalg.eigvalsh(H)  # eigvalsh for symmetric H
    return bool(np.all(eigenvalues >= 0))

H_convex = np.array([[2., 1.], [1., 3.]])   # hand-derived eigenvalues (5 +/- sqrt 5)/2
H_saddle = np.array([[1., 2.], [2., 1.]])   # hand-derived eigenvalues -1 and 3

for label, H in [("H_convex", H_convex), ("H_saddle", H_saddle)]:
    eigs = np.linalg.eigvalsh(H)
    print("{}: eigenvalues = {}  ->  convex = {}".format(
        label, np.round(eigs, 4), is_convex_quadratic(H)))


## Visualizing different critical points

In [ ]:
x = np.linspace(-2, 2, 100)
y = np.linspace(-2, 2, 100)
X, Y = np.meshgrid(x, y)

functions = {
    'Local minimum\n(H positive definite)': X**2 + Y**2,
    'Saddle point\n(H indefinite)':          X**2 - Y**2,
    'Non-convex\n(multiple minima)':         np.sin(3*X) * np.cos(3*Y),
}

fig, axes = plt.subplots(1, 3, figsize=(16, 5), subplot_kw={'projection': '3d'})

for ax, (title, Z) in zip(axes, functions.items()):
    surf = ax.plot_surface(X, Y, Z, cmap='twilight', alpha=0.85,
                           linewidth=0, antialiased=True)
    ax.set_title(title, pad=10, fontsize=10)
    ax.set_xlabel('w₁'); ax.set_ylabel('w₂'); ax.set_zlabel('Loss')
    ax.tick_params(labelsize=7)

plt.suptitle('Loss Landscape Shapes', y=1.02, fontsize=13)
plt.tight_layout(); plt.show()

## Effect of learning rate on convergence

In [ ]:
# Quadratic f(w) = 2 w1^2 + 0.5 w2^2, so H = diag(4, 1): lambda_max = 4, lambda_min = 1.
def loss(w): return 2*w[0]**2 + 0.5*w[1]**2
def grad(w): return np.array([4*w[0], w[1]])

H_lr = np.array([[4., 0.], [0., 1.]])
lam_max = np.max(np.linalg.eigvalsh(H_lr))
eta_opt = 1.0 / lam_max          # fastest stable, balanced step
eta_diverge = 2.0 / lam_max      # iteration diverges above this threshold
print("lambda_max = {:.1f}  ->  eta* = 1/L = {:.3f},  diverges when eta > 2/L = {:.3f}".format(
    lam_max, eta_opt, eta_diverge))

learning_rates = [0.05, 0.2, 0.49, 0.51]  # last two straddle the 0.5 threshold
w0 = np.array([2.0, 2.0])

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

xx, yy = np.meshgrid(np.linspace(-2.5, 2.5, 200), np.linspace(-2.5, 2.5, 200))
Z = 2*xx**2 + 0.5*yy**2

for ax, lr in zip(axes, learning_rates):
    path = [w0.copy()]
    w = w0.copy()
    for _ in range(40):
        w = w - lr * grad(w)
        path.append(w.copy())
        if np.any(np.abs(w) > 100):
            break
    path = np.array(path)

    ax.contourf(xx, yy, Z, levels=15, cmap='twilight', alpha=0.6)
    ax.contour(xx, yy, Z, levels=15, colors='white', alpha=0.2, linewidths=0.5)
    ax.plot(path[:, 0], path[:, 1], 'o-', color='#f97316', ms=4, lw=1.5)
    ax.scatter(*path[0], color='#2dd4bf', s=80, zorder=5, label='Start')
    ax.scatter(0, 0, marker='*', color='#f59e0b', s=150, zorder=5, label='Min')
    ax.set_xlim(-2.5, 2.5); ax.set_ylim(-2.5, 2.5)
    ax.set_aspect('equal'); ax.grid(False)
    final_loss = loss(path[-1])
    status = 'DIVERGED' if final_loss > 100 else 'Loss={:.3f}'.format(final_loss)
    ax.set_title('eta = {}\n{}'.format(lr, status), fontsize=10)

plt.suptitle('Effect of Learning Rate (optimal eta = 1/L = 0.25, diverges for eta > 0.5)',
             y=1.04, fontsize=12)
plt.tight_layout(); plt.show()

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise 1 — The gradient of the worked example

Differentiate the lesson's function $f(x, y) = x^2 + xy + y^2 - 3x$ partial by partial:

$$\nabla f = \begin{bmatrix} \partial f / \partial x \\ \partial f / \partial y \end{bmatrix}
= \begin{bmatrix} 2x + y - 3 \\ x + 2y \end{bmatrix}$$

The checks verify the critical point $(2, -1)$ from the lesson and cross-check against finite differences at several other points.

In [ ]:
def grad_f(x, y):
    """Gradient of f(x, y) = x^2 + x*y + y^2 - 3x as a length-2 array."""
    # TODO(you): partial wrt x: 2x + y - 3
    dfdx = ...

    # TODO(you): partial wrt y: x + 2y
    dfdy = ...

    return np.array([dfdx, dfdy])

In [ ]:
# Checks — run me
assert np.allclose(grad_f(2, -1), [0, 0]), "(2, -1) is the critical point from the lesson"

f = lambda x, y: x ** 2 + x * y + y ** 2 - 3 * x
h = 1e-6
for (px, py) in [(0.0, 0.0), (1.0, 2.0), (-3.0, 0.5)]:
    nx = (f(px + h, py) - f(px - h, py)) / (2 * h)
    ny = (f(px, py + h) - f(px, py - h)) / (2 * h)
    assert np.allclose(grad_f(px, py), [nx, ny], atol=1e-5), f"gradient mismatch at {(px, py)}"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def grad_f(x, y):
    dfdx = 2 * x + y - 3
    dfdy = x + 2 * y
    return np.array([dfdx, dfdy])
```

</details>

### Exercise 2 — Classify critical points by curvature

At a critical point the gradient is zero — the **Hessian's eigenvalues** decide what kind of point it is:

- all $\lambda_i > 0$ → curves up in every direction → **minimum**
- all $\lambda_i < 0$ → curves down in every direction → **maximum**
- mixed signs → up one way, down another → **saddle**

Implement the test with `np.linalg.eigvalsh` (the symmetric-matrix eigensolver). The last check is the classic trap: a Hessian with *positive diagonal* can still be a saddle once the off-diagonals get big.

In [ ]:
def classify_critical_point(H):
    """Return 'minimum', 'maximum', or 'saddle' from the Hessian H (symmetric)."""
    # TODO(you): eigenvalues of H (hint: np.linalg.eigvalsh)
    lams = ...

    # TODO(you): all positive -> minimum; all negative -> maximum; otherwise saddle
    ...

In [ ]:
# Checks — run me
assert classify_critical_point([[2, 1], [1, 2]]) == "minimum", "eigenvalues 1, 3 -> bowl"
assert classify_critical_point([[-2, 0], [0, -3]]) == "maximum", "eigenvalues -2, -3 -> dome"
assert classify_critical_point([[2, 0], [0, -1]]) == "saddle", "mixed signs -> saddle"
assert classify_critical_point([[1, 2], [2, 1]]) == "saddle", "positive diagonal yet eigenvalues -1, 3"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def classify_critical_point(H):
    lams = np.linalg.eigvalsh(np.asarray(H, dtype=float))
    if np.all(lams > 0):
        return "minimum"
    if np.all(lams < 0):
        return "maximum"
    return "saddle"
```

</details>